In [ ]:
from datasets import load_dataset

In [29]:
from datasets import load_dataset

data_files = {
    "train": "https://huggingface.co/datasets/QCRI/HumAID-all/resolve/main/data/train-00000-of-00001.parquet",
    "validation": "https://huggingface.co/datasets/QCRI/HumAID-all/resolve/main/data/validation-00000-of-00001.parquet",
    "test": "https://huggingface.co/datasets/QCRI/HumAID-all/resolve/main/data/test-00000-of-00001.parquet",
}

ds = load_dataset("parquet", data_files=data_files)

In [30]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['tweet_text', 'class_label'],
        num_rows: 53531
    })
    validation: Dataset({
        features: ['tweet_text', 'class_label'],
        num_rows: 7793
    })
    test: Dataset({
        features: ['tweet_text', 'class_label'],
        num_rows: 15160
    })
})


In [52]:
import re
def clean_text(text):
    # Normalize whitespace
    text = ' '.join(text.split())
    # Replace URLs with <URL>
    text = re.sub(r'http\S+|www\S+|https\S+', '<URL>', text, flags=re.MULTILINE)
    # Replace mentions with <MENTION>
    text = re.sub(r'@\w+', '<MENTION>', text)
    # Replace &amp; with &
    text = text.replace('&amp;', '&')

    return text

In [53]:
training_texts = ds["train"]["tweet_text"]
cleaned_training_texts = [clean_text(text) for text in training_texts]


In [54]:
print(cleaned_training_texts[:5])  

['Powerful Ecuador quake kills at least 235: PORTOVIEJO - Rescuers in Ecuador raced to dig out people trapped un', 'Im at awe and saddened with the #EcuadorEarthquake Please any donations can help the most unfortunate!', 'RT <MENTION>: Our hearts are with everyone in Kumamoto and Ecuador whos been affected by these earthquakes. :( This is has been a terr', 'RT <MENTION>: Ecuador quake death toll has risen to 233. Correa just confirmed. #pedernales #ecuadorlisto', 'RT <MENTION>: BREAKING PHOTOS: Major damage reported in Ecuador after massive earthquake.']


In [55]:
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.models import BPE  
from tokenizers.trainers import BpeTrainer

In [60]:
# Set up the tokenizer with BPE and specify the unknown token
tokenizer = Tokenizer(BPE(unknown_token="<UNK>"))

# Need this line to ensure that the tokenizer splits on whitespace before applying BPE
# E.g., "Hello world" becomes ["Hello", "world"] before BPE is applied
tokenizer.pre_tokenizer = Whitespace()

# Chose vocab size of 16000 and added special tokens for padding, unknown, classification, separation, and masking
tokenizer_trainer = BpeTrainer(vocab_size=16000,special_tokens=["<PAD>", "<UNK>", "<SOS>", "<EOS>","<URL>", "<MENTION>"])

tokenizer.train_from_iterator(cleaned_training_texts, trainer=tokenizer_trainer)

In [61]:
for text in cleaned_training_texts[:5]:
    output = tokenizer.encode(text)

    print("\nOriginal:", text)
    print("Tokens:  ", output.tokens)


Original: Powerful Ecuador quake kills at least 235: PORTOVIEJO - Rescuers in Ecuador raced to dig out people trapped un
Tokens:   ['Powerful', 'Ecuador', 'quake', 'kills', 'at', 'least', '235', ':', 'PORT', 'O', 'VI', 'E', 'JO', '-', 'Rescuers', 'in', 'Ecuador', 'rac', 'ed', 'to', 'dig', 'out', 'people', 'trapped', 'un']

Original: Im at awe and saddened with the #EcuadorEarthquake Please any donations can help the most unfortunate!
Tokens:   ['Im', 'at', 'awe', 'and', 'saddened', 'with', 'the', '#', 'EcuadorEarthquake', 'Please', 'any', 'donations', 'can', 'help', 'the', 'most', 'unfortunate', '!']

Original: RT <MENTION>: Our hearts are with everyone in Kumamoto and Ecuador whos been affected by these earthquakes. :( This is has been a terr
Tokens:   ['RT', '<MENTION>', ':', 'Our', 'hearts', 'are', 'with', 'everyone', 'in', 'Kum', 'amoto', 'and', 'Ecuador', 'whos', 'been', 'affected', 'by', 'these', 'earthquakes', '.', ':(', 'This', 'is', 'has', 'been', 'a', 'terr']

Original: RT <

In [62]:
# check the lengths of the tokenized sequences
lengths = []

for text in cleaned_training_texts:
    output = tokenizer.encode(text)
    lengths.append(len(output.ids))

print("Max:", max(lengths))
print("Average:", sum(lengths) / len(lengths))

Max: 158
Average: 29.06670900973268


In [63]:
from tokenizers.processors import TemplateProcessing

tokenizer.post_processor = TemplateProcessing(
    # $A Put the encoded tweet text here
    single="<SOS> $A <EOS>",

    # Tells tokenizer what is start and end of the sequence
    special_tokens=[("<SOS>", tokenizer.token_to_id("<SOS>")), ("<EOS>", tokenizer.token_to_id("<EOS>"))],
)

# Got to limit the length of the sequences to avoid memory issues during training.
tokenizer.enable_truncation(max_length=128)  # Truncate sequences to a maximum length of 128 tokens

# Padding
tokenizer.enable_padding(pad_id=tokenizer.token_to_id("<PAD>"), pad_token="<PAD>")

In [65]:
for text in cleaned_training_texts[:5]:
    output = tokenizer.encode(text)

    print("\nOriginal:", text)
    print("Tokens:  ", output.tokens)
    print("IDs:     ", output.ids)
    print("Attention Mask: ", output.attention_mask)


Original: Powerful Ecuador quake kills at least 235: PORTOVIEJO - Rescuers in Ecuador raced to dig out people trapped un
Tokens:   ['<SOS>', 'Powerful', 'Ecuador', 'quake', 'kills', 'at', 'least', '235', ':', 'PORT', 'O', 'VI', 'E', 'JO', '-', 'Rescuers', 'in', 'Ecuador', 'rac', 'ed', 'to', 'dig', 'out', 'people', 'trapped', 'un', '<EOS>']
IDs:      [2, 4840, 1406, 1000, 2663, 879, 1398, 9475, 29, 4182, 50, 2922, 40, 12734, 16, 5174, 872, 1406, 5800, 888, 880, 5792, 996, 1007, 3409, 938, 3]
Attention Mask:  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Original: Im at awe and saddened with the #EcuadorEarthquake Please any donations can help the most unfortunate!
Tokens:   ['<SOS>', 'Im', 'at', 'awe', 'and', 'saddened', 'with', 'the', '#', 'EcuadorEarthquake', 'Please', 'any', 'donations', 'can', 'help', 'the', 'most', 'unfortunate', '!', '<EOS>']
IDs:      [2, 1645, 879, 14098, 901, 5945, 984, 883, 7, 2840, 1192, 1211, 1405, 1059, 967, 883, 1498, 

In [69]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [71]:
tokenizer.save("/content/drive/MyDrive/tokenizer/humanitarian_bpe_tokenizer.json")

In [ ]:
# If saving to local machine:
tokenizer.save("humanitarian_bpe_tokenizer.json")